In [1]:
import pandas as pd

In [4]:
df = pd.read_csv('../data/unprocessed/all_data.csv')
df.shape

(1999516, 46)

In [14]:
df

,id,comment_text,split,created_date,publication_id,parent_id,article_id,rating,funny,wow,...,white,asian,latino,other_race_or_ethnicity,physical_disability,intellectual_or_learning_disability,psychiatric_or_mental_illness,other_disability,identity_annotator_count,toxicity_annotator_count
0,1083994,He got his money... now he lies in wait till a...,train,2017-03-06 15:21:53.675241+00,21,NaN,317120,approved,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,67
1,650904,Mad dog will surely put the liberals in mental...,train,2016-12-02 16:44:21.329535+00,21,NaN,154086,approved,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,76
2,5902188,And Trump continues his lifelong cowardice by ...,train,2017-09-05 19:05:32.341360+00,55,NaN,374342,approved,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,63
3,7084460,"""while arresting a man for resisting arrest"".\...",test,2016-11-01 16:53:33.561631+00,13,NaN,149218,approved,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,76
4,5410943,Tucker and Paul are both total bad ass mofo's.,train,2017-06-14 05:08:21.997315+00,21,NaN,344096,approved,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,80
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1999511,1018736,Another man shamming article. If white men did...,train,2017-02-20 07:20:49.964620+00,54,NaN,169202,approved,0,0,...,0.8,0.0,0.0,0.0,0.000000,0.0,0.0,0.00000,10,10
1999512,340016,"""no matter what is put in front of you regardi...",train,2016-06-06 06:43:04.780968+00,21,339965.0,137961,approved,0,0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.00000,10,10
1999513,919629,The Democrat party aided and abetted by it's M...,train,2017-01-30 02:44:29.168863+00,54,NaN,164845,rejected,0,1,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.00000,11,10
1999514,5165492,I just don't find her a very good representati...,train,2017-04-22 18:42:02.442987+00,54,NaN,328877,approved,1,0,...,0.0,0.0,0.0,0.0,0.003717,0.0,0.0,0.00000,269,10


## Test comments by minimum annotator count

Detoxify was trained on the train split, so the sample comes from the test split. Counts are deduplicated by comment text and split into four human-toxicity bands.

In [ ]:
test = df[df.split == 'test'].assign(
    text=lambda d: d.comment_text.str.strip(),
    band=lambda d: pd.cut(d.toxicity, [0, 0.25, 0.5, 0.75, 1.01], right=False,
                          labels=['tox_0_to_0.25', 'tox_0.25_to_0.5', 'tox_0.5_to_0.75', 'tox_0.75_to_1']),
)

rows = []
for k in range(10, 101, 10):
    eligible = test[test.toxicity_annotator_count >= k].drop_duplicates('text')
    rows.append({'min_annotators': k, **eligible.band.value_counts().to_dict()})

threshold_counts = pd.DataFrame(rows)[['min_annotators', 'tox_0_to_0.25', 'tox_0.25_to_0.5', 'tox_0.5_to_0.75', 'tox_0.75_to_1']]
threshold_counts['total'] = threshold_counts.iloc[:, 1:].sum(axis=1)
threshold_counts.to_csv('../data/unprocessed/test_annotator_threshold_counts.csv', index=False)
threshold_counts

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

series = [('tox_0_to_0.25', 'Toxicity 0–0.25', '#2a78d6'),
          ('tox_0.25_to_0.5', 'Toxicity 0.25–0.5', '#eb6834'),
          ('tox_0.5_to_0.75', 'Toxicity 0.5–0.75', '#1baf7a'),
          ('tox_0.75_to_1', 'Toxicity 0.75–1', '#eda100')]
ink, ink2, muted, grid, base = '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7'
plt.rcParams.update({'font.family': ['Helvetica Neue', 'Arial', 'sans-serif'], 'font.size': 11})

fig, ax = plt.subplots(figsize=(9, 5.5), dpi=200)
fig.patch.set_facecolor('#fcfcfb'); ax.set_facecolor('#fcfcfb')
for col, label, color in series:
    d = threshold_counts[threshold_counts[col] > 0]  # log scale can't show zero
    ax.plot(d.min_annotators, d[col], color=color, lw=2, marker='o', ms=5.5,
            mec='#fcfcfb', mew=1.5, label=label, zorder=3)
    ax.annotate(label, (d.min_annotators.iloc[-1], d[col].iloc[-1]), xytext=(8, 0),
                textcoords='offset points', va='center', color=ink2, fontsize=9.5)

ax.axhline(25, color=muted, lw=1, ls=(0, (4, 3)), zorder=2)
ax.text(10, 25 * 1.12, '25 per band needed for a 100-comment sample', color=ink2, fontsize=9)

ax.set_yscale('log')
ax.set_ylim(0.7, 25000)
ax.set_xlim(7, 122)
ax.set_xticks(threshold_counts.min_annotators)
ax.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
ax.yaxis.set_minor_locator(matplotlib.ticker.NullLocator())
ax.grid(axis='y', color=grid, lw=0.8); ax.set_axisbelow(True)
for s in ['top', 'right', 'left']: ax.spines[s].set_visible(False)
ax.spines['bottom'].set_color(base)
ax.tick_params(which='both', colors=muted, length=0, labelcolor=ink2)
ax.set_xlabel('Minimum toxicity annotators per comment', color=ink2)
ax.set_ylabel('Test comments (log scale, deduplicated)', color=ink2)
ax.set_title('Test comments remaining by minimum number of annotators',
             loc='left', color=ink, fontsize=13, fontweight='bold', pad=14)
leg = ax.legend(loc='lower left', frameon=False, fontsize=9.5)
for t in leg.get_texts(): t.set_color(ink2)
fig.text(0.01, 0.01, 'Civil Comments test split. The 0.25–0.5 band has 0 comments at ≥100 annotators (not shown on the log scale).',
         color=muted, fontsize=8)
fig.tight_layout(rect=(0, 0.03, 1, 1))
fig.savefig('../data/unprocessed/test_annotator_threshold_counts.png', facecolor=fig.get_facecolor())
plt.show()